# Import and init

In [ ]:
import datetime
from time import sleep
import time
import csv
import re
import os

import requests
from selenium import webdriver
import chromedriver_autoinstaller
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
import undetected_chromedriver as uc # !pip install undetected-chromedriver

import project_secrets


In [ ]:
DATE_NOW = datetime.datetime.now().strftime('%y%m%d')
DATE_NOW

# Selenium

In [ ]:
# Initialise webdriver object
driver = uc.Chrome(version_main = 145)
driver.delete_all_cookies()

# Load web page
driver.get("https://app.musicleague.com/home/")
driver.maximize_window()

# Ensure it loads
sleep(2)
# Localise and press button "Login with Spotify"
driver.find_element(
    By.CSS_SELECTOR, 
    ".btn.border-2.btn-outline-light.fw-semibold"
    ).click()
sleep(2)

# Localise the "Email" field
wait = WebDriverWait(driver, 10)
search_box = wait.until(
    EC.presence_of_element_located((
        By.CSS_SELECTOR, 
        ".e-91185-form-input.e-91185-baseline.e-91185-form-control.encore-text-body-medium.e-91185-focus-border"
        ))
)
# Write my email there
search_box.send_keys(project_secrets.email_auth)
# Localise and click the "Continue" button
search_button = driver.find_element(
    By.CSS_SELECTOR, 
    ".e-91185-baseline.e-91185-overflow-wrap-anywhere.e-91185-button-primary__inner.encore-bright-accent-set.e-91185-button--medium"
    )
search_button.click()

# At this point, I manually input the 6-digit code that I 
# received on my email, and then press log in button.
# As soon as i finish this step, I type 'yes' to continue with the automatic
# part of the program.
a = input('let me know when you are done')
if a.lower() == 'yes':
    print('continuing')

# Get page source
html = driver.page_source

# Locate the name of the league 
element = driver.find_element(
    By.XPATH, 
    "//a[text()='all bangers all the time ✨']"
    )
# Scroll it into view and press
driver.execute_script("arguments[0].scrollIntoView()", element)
driver.find_element(
    By.XPATH, 
    "//a[text()='all bangers all the time ✨']"
).click()
sleep(5)

# Find all link html objects on the page (there is no pagination for results,
# so it's easy to just get all elements from the page)
links = driver.find_elements(
    By.CSS_SELECTOR, 
    ".stretched-link.text-body-tertiary.text-decoration-none"
    )
urls = [link.get_attribute('href') for link in links]
urls = [i for i in urls if 'app.musicleague.com' in i]

# Process data from each url
data = []
for counter, url in enumerate(urls, start = 1):
    driver.get(url)
    wait.until(
        EC.presence_of_element_located((
            By.CSS_SELECTOR, 
            ".card-footer.bg-body.collapse.show"
        ))
    )
    # Extract all visible text
    text = driver.find_element(
        By.TAG_NAME, 
        'body'
        ).text
    # store result
    data.append([url, text])
    
    # save result from each round to html
    html = driver.page_source
    with open(f'output/scraped_data/page_{counter}.html', 'w', encoding = 'utf-8') as f:
        f.write(html)
        
    driver.back()
    wait.until(
        EC.presence_of_all_elements_located((
            By.CSS_SELECTOR, 
            ".stretched-link.text-body-tertiary.text-decoration-none"
            ))
    )
#    time.sleep(1)

# save all data from all rounds to one single csv
with open(f'output/scraped_data/{DATE_NOW}_scraped_pages.csv', 'w', newline = '', encoding = 'utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['url', 'text'])
    writer.writerows(data)
    writer.writerows('\n\n==========================\n\n')
       